# GitHub · Hugging Face · Kaggle 미니 실습

INSIGHT 15기 · 플랫폼 기초

이 실습의 목표는 세 플랫폼에서 무엇을 가져오는지 직접 확인하는 것입니다.

| 플랫폼 | 이번에 가져올 것 | 해볼 일 |
|---|---|---|
| GitHub | 코드와 변경 이력 | 저장소 복제 후 파일·커밋 확인 |
| Hugging Face | 학습된 모델 가중치 | 감정 분석 추론 |
| Kaggle | 데이터셋 | CSV 다운로드 후 기초 탐색 |

예상 시간은 15~20분입니다. 공개 자료만 사용하므로 계정이나 API 토큰은 필요하지 않습니다.

## 0. Colab 준비

Colab 메뉴에서 `런타임 → 런타임 유형 변경 → T4 GPU`를 선택해도 좋습니다.
Hugging Face 추론은 GPU가 있으면 자동으로 GPU를 사용하고, 없으면 CPU로도 실행됩니다.

In [1]:
import importlib.util
import subprocess
import sys

packages = {
    'transformers': 'transformers',
    'kagglehub': 'kagglehub',
}
for module_name, package_name in packages.items():
    if importlib.util.find_spec(module_name) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package_name])

import pandas as pd
import torch

device_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('실행 장치:', device_name)
print('준비 완료')

실행 장치: CPU
준비 완료


---
## 1. GitHub — 코드를 내 작업 공간으로 가져오기

`git clone`은 GitHub 저장소의 파일과 변경 이력을 현재 작업 공간에 복사합니다.
여기서는 GitHub의 공개 예제 저장소를 복제합니다.

In [2]:
from pathlib import Path

repo_url = 'https://github.com/octocat/Hello-World.git'
repo_dir = Path('Hello-World')

if not repo_dir.exists():
    subprocess.run(['git', 'clone', repo_url], check=True)
else:
    print('이미 복제되어 있어 기존 폴더를 사용합니다.')

print('저장소 위치:', repo_dir.resolve())
print('파일 목록:', [p.name for p in repo_dir.iterdir() if p.name != '.git'])

Cloning into 'Hello-World'...


저장소 위치: /Users/damisoda/Documents/Codex/2026-08-21/new-chat/outputs/Hello-World
파일 목록: ['README']


In [3]:
readme_path = repo_dir / 'README'
print('[README 내용]')
print(readme_path.read_text(encoding='utf-8'))

print('[최근 커밋 5개]')
result = subprocess.run(
    ['git', '-C', str(repo_dir), 'log', '--oneline', '-5'],
    check=True, capture_output=True, text=True
)
print(result.stdout)

[README 내용]
Hello World!

[최근 커밋 5개]
7fd1a60 Merge pull request #6 from Spaceghost/patch-1
7629413 New line at end of file. --Signed off by Spaceghost
553c207 first commit



### 확인 문제

1. 저장소에서 실제 코드나 문서는 어느 폴더에 생겼나요?
2. `git log`에는 파일 내용이 아니라 무엇이 기록되어 있나요?
3. `clone`과 웹페이지에서 파일을 한 개 내려받는 것은 무엇이 다른가요?

**정리:** GitHub에서 가져오는 것은 파일만이 아니라 프로젝트 구조와 Git 변경 이력입니다.

---
## 2. Hugging Face — 학습이 끝난 모델 사용하기

이번에는 `distilbert-base-uncased-finetuned-sst-2-english` 모델을 사용합니다.
이미 영어 문장의 긍정·부정을 분류하도록 학습된 모델이므로, 우리는 새로 학습시키지 않고 추론만 합니다.
첫 실행에서는 모델 파일을 내려받느라 잠시 걸릴 수 있습니다.

In [4]:
from transformers import pipeline

model_id = 'distilbert/distilbert-base-uncased-finetuned-sst-2-english'
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline(
    task='text-classification',
    model=model_id,
    device=device,
)
print('모델 준비 완료:', model_id)

/Users/damisoda/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 20541.92it/s]


모델 준비 완료: distilbert/distilbert-base-uncased-finetuned-sst-2-english


In [5]:
sentences = [
    'The class was clear and surprisingly fun.',
    'The instructions were confusing and frustrating.',
    'The model worked, but it was slower than I expected.',
]
predictions = classifier(sentences, truncation=True)

hf_result = pd.DataFrame({
    'sentence': sentences,
    'label': [p['label'] for p in predictions],
    'confidence': [round(p['score'], 4) for p in predictions],
})
hf_result

,sentence,label,confidence
0,The class was clear and surprisingly fun.,POSITIVE,0.9999
1,The instructions were confusing and frustrating.,NEGATIVE,0.9995
2,"The model worked, but it was slower than I exp...",NEGATIVE,0.9908


### 직접 해보기

아래 문장을 본인의 영어 문장으로 바꾸고 결과를 확인하세요. 이 모델은 영어 데이터로 학습되었으므로 영어 문장을 사용합니다.

In [6]:
my_sentence = 'I enjoyed learning how pretrained models work.'  # 이 문장을 바꾸세요.
my_prediction = classifier(my_sentence)[0]
print('문장:', my_sentence)
print('예측:', my_prediction['label'])
print('확신도:', round(my_prediction['score'], 4))

문장: I enjoyed learning how pretrained models work.
예측: POSITIVE
확신도: 0.9992


### 확인 문제

1. 우리가 직접 학습한 것은 무엇인가요?
2. 코드의 `model_id`는 Hugging Face에서 무엇을 가리키나요?
3. `confidence`가 높다는 것은 모델의 예측이 반드시 옳다는 뜻인가요?

**정리:** Hugging Face에서 가져오는 핵심은 다른 사람이 학습해 둔 모델 구조·가중치와 사용에 필요한 설정입니다.

---
## 3. Kaggle — 공개 데이터셋 가져와 살펴보기

KaggleHub로 공개 Spotify 데이터의 CSV 파일을 내려받습니다.
공개 데이터셋은 보통 로그인 없이 받을 수 있습니다. 토큰이 필요한 상황이 생겨도 토큰 값을 노트북 셀에 직접 적지는 마세요.

In [7]:
import kagglehub

csv_path = kagglehub.dataset_download(
    'bricevergnou/spotify-recommendation',
    path='data.csv',
)
print('다운로드된 파일:', csv_path)

spotify = pd.read_csv(csv_path)
print('데이터 크기:', spotify.shape)
spotify.head()

100%|██████████| 14.0k/14.0k [00:00<00:00, 9.27MB/s]

다운로드된 파일: /Users/damisoda/.cache/kagglehub/datasets/bricevergnou/spotify-recommendation/versions/2/data.csv
데이터 크기: (195, 14)


,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_ms,time_signature,liked
0,0.803,0.6240,7,-6.764,0,0.0477,0.451,0.000734,0.1000,0.6280,95.968,304524,4,0
1,0.762,0.7030,10,-7.951,0,0.3060,0.206,0.000000,0.0912,0.5190,151.329,247178,4,1
2,0.261,0.0149,1,-27.528,1,0.0419,0.992,0.897000,0.1020,0.0382,75.296,286987,4,0
3,0.722,0.7360,3,-6.994,0,0.0585,0.431,0.000001,0.1230,0.5820,89.860,208920,4,1
4,0.787,0.5720,1,-7.516,1,0.2220,0.145,0.000000,0.0753,0.6470,155.117,179413,4,1


In [8]:
print('[열 이름]')
print(spotify.columns.tolist())

print('\n[결측치가 많은 열 상위 5개]')
print(spotify.isna().sum().sort_values(ascending=False).head())

numeric = spotify.select_dtypes(include='number')
print('\n[수치형 열 요약]')
numeric.describe().T.head()

[열 이름]
['danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'liked']

[결측치가 많은 열 상위 5개]
danceability    0
energy          0
key             0
loudness        0
mode            0
dtype: int64

[수치형 열 요약]


,count,mean,std,min,25%,50%,75%,max
danceability,195.0,0.636656,0.216614,0.1300,0.4625,0.705,0.7990,0.946
energy,195.0,0.638431,0.260096,0.0024,0.5335,0.659,0.8375,0.996
key,195.0,5.497436,3.415209,0.0000,2.0000,6.000,8.0000,11.000
loudness,195.0,-9.481631,6.525086,-42.2610,-9.9620,-7.766,-5.8290,-2.336
mode,195.0,0.538462,0.499802,0.0000,0.0000,1.000,1.0000,1.000


### 직접 해보기

수치형 열 중 하나를 골라 평균, 최솟값, 최댓값을 출력하세요. 먼저 위 셀에서 실제 열 이름을 확인합니다.

In [9]:
chosen_column = numeric.columns[0]  # 원하는 수치형 열 이름으로 바꿔도 됩니다.

print('선택한 열:', chosen_column)
print('평균:', spotify[chosen_column].mean())
print('최솟값:', spotify[chosen_column].min())
print('최댓값:', spotify[chosen_column].max())

선택한 열: danceability
평균: 0.6366564102564103
최솟값: 0.13
최댓값: 0.946


### 확인 문제

1. 데이터는 몇 행, 몇 열인가요?
2. 결측치가 있는 열이 있나요?
3. 이 데이터로 예측 문제를 만든다면 무엇을 입력과 정답으로 둘 수 있을까요?

**정리:** Kaggle에서 가져오는 것은 실제 분석·학습에 사용할 데이터와 다른 사용자의 분석 사례입니다.

---
## 4. 마무리

| 오늘 실행한 코드 | 의미 |
|---|---|
| `git clone ...` | GitHub의 코드와 변경 이력을 복제 |
| `pipeline(..., model=model_id)` | Hugging Face의 학습된 모델을 불러와 추론 |
| `kagglehub.dataset_download(...)` | Kaggle의 데이터 파일을 다운로드 |

세 플랫폼의 공통점은 이미 공개된 결과물을 가져와 확인하고, 그 위에서 내 분석이나 프로젝트를 시작할 수 있다는 점입니다.

공식 참고 자료: [GitHub 저장소 복제](https://docs.github.com/en/repositories/creating-and-managing-repositories/cloning-a-repository) · [Hugging Face pipeline](https://huggingface.co/docs/transformers/pipeline_tutorial) · [KaggleHub](https://github.com/Kaggle/kagglehub)